# Demo: Create Agents to Research and Write an Article

This notebook demonstrates how to create a simple multi-agent multi-agent system using the crewAI framework.

## Preparing environment

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from crewai import Agent, Task, Crew, LLM

In [3]:
from IPython.display import Markdown

In [4]:
AGENT_ALLOW_DELEGATION = False
AGENT_VERBOSE = True
AGENT_TRACING = True

## Prepare locally hosted LLM

In [5]:
ollama_llm = LLM(
    model="ollama/mistral:latest",
    base_url="http://localhost:11434",
    temperature=0.7,
)

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_:

```
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:

```
varname = """line 1 of text
             line 2 of text
          """
```

is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [6]:
planner = Agent(
    role="Connect Planner",
    goal="Plan engaging and facturally accurate content on {topic}",
    backstory="You're working on planning a blog article"
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions."
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    llm=ollama_llm,
    allow_delegation=AGENT_ALLOW_DELEGATION,
    verbose=AGENT_VERBOSE
)

### Agent: Writer

In [7]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provided by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    llm=ollama_llm,
    allow_delegation=AGENT_ALLOW_DELEGATION,
    verbose=AGENT_VERBOSE
)

### Agent: Editor

In [8]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    llm=ollama_llm,
    allow_delegation=AGENT_ALLOW_DELEGATION,
    verbose=AGENT_VERBOSE
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [9]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
                    "with an outline, audience analysis, "
                    "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [10]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
        "3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
                    "in markdown format, ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [11]:
edit = Task(
    description=(
        "Proofread the given blog post for "
        "grammatical errors and alignment with the brand's voice."
    ),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: _For this simple example_, the tasks will be performed sequentially (i.e. they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=True` allows you to see execution logs.



In [12]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=AGENT_VERBOSE,
    tracing=AGENT_TRACING
)

## Running the Crew

In [13]:
from time import perf_counter

In [14]:
start = perf_counter()
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})
end = perf_counter()
print(f"\n\n{'='*100}\nTasks are finished in {end - start} seconds.\n{'='*100}\n\n")

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: feb721e7-d204-4d8a-b557-a869ea16228c                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Connect Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Connect Planner                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Content Plan Document for "Exploring the Latest Trends in Artificial Intelligence"                             │
│                                                                                                                 │
│  Audience Analysis:                                                                                             │
│  Target audience for this article includes professionals, business owners, and tech enthusiasts who are         │
│  interested in understanding the latest developments in AI technology, its impact on various industries, and    │
│  how they can leverage it for their businesses or careers. The pain points of our audience may include a lack   │
│  of understanding about AI technology, concerns about job displacement due to automation, and a desire to stay  │
│  up-to-date with the latest trends.                                                                             │
│                                                                                                                 │
│  Content Outline:                                                                                               │
│                                                                                                                 │
│  1. Introduction:                                                                                               │
│  * Brief overview of Artificial Intelligence and its importance in today's world.                               │
│  * Hook for readers by highlighting some recent news or developments in AI.                                     │
│  1. Latest Trends in AI:                                                                                        │
│  * Explain the current state of AI technology, including key advancements and breakthroughs.                    │
│  * Discuss emerging trends such as AI-powered healthcare, autonomous vehicles, and AI ethics.                   │
│  * Highlight some notable examples or case studies that illustrate these trends in action.                      │
│  1. Key Players in the AI Industry:                                                                             │
│  * Introduce some of the key players shaping the future of AI, including companies like Google, Tesla, and      │
│  IBM.                                                                                                           │
│  * Discuss their contributions to the field and their impact on the industry.                                   │
│  1. Impact of AI on Various Industries:                                                                         │
│  * Explain how AI is being used across different industries such as healthcare, finance, manufacturing, and     │
│  retail.                                                                                                        │
│  * Highlight some of the benefits and challenges that these industries are facing due to AI adoption.           │
│  1. The Future of AI:                                                                                           │
│  * Discuss potential future developments in AI technology, including advancements in machine learning, natural  │
│  language processing, and robotics.                                                                             │
│  * Explore the ethical considerations surrounding AI, s

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 85a8a6ae-5740-42c5-9711-00d16afd9306                                                                     │
│  Agent: Connect Planner                                                                                         │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Task: 1. Use the content plan to craft a compelling blog post on Artificial Intelligence.                      │
│  2. Incorporate SEO keywords naturally.                                                                         │
│  3. Sections/Subtitles are properly named in an engaging manner.                                                │
│  4. Ensure the post is structured with an engaging introduction, insightful body, and a summarizing             │
│  conclusion.                                                                                                    │
│  5. Proofread for grammatical errors and alignment with the brand's voice.                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  Artificial Intelligence: Navigating the New Wave of Innovation 🌐💡🔬                                          │
│  ===================================================================                                            │
│                                                                                                                 │
│  Welcome to the cutting edge of technological evolution! Today, we're diving deep into the world of Artificial  │
│  Intelligence (AI), a realm brimming with potential that promises to revolutionize industries and shape our     │
│  future. So buckle up as we traverse the latest trends, key players, impactful applications, and ethical        │
│  considerations in AI technology.                                                                               │
│                                                                                                                 │
│  🌐 **Latest Trends in AI** 🚀💻                                                                                │
│  -----------------------------                                                                                  │
│                                                                                                                 │
│  The AI landscape is evolving at an unprecedented pace, with breakthroughs and advancements that are reshaping  │
│  the way we live and work. From AI-powered healthcare systems that diagnose diseases faster than ever before    │
│  to autonomous vehicles cruising our streets, the impact of AI is becoming increasingly palpable. Here's a      │
│  glimpse at some emerging trends that are steering the course for this transformative technology:               │
│                                                                                                                 │
│  - **AI-Powered Healthcare:** Medical professionals are now leveraging AI to sift through massive amounts of    │
│  data and make accurate diagnoses faster than ever before. The potential applications for this technology in    │
│  the healthcare industry are vast, ranging from personalized treatment plans to predictive analytics that       │
│  could save countless lives. [Source: AI in Healthcare Market Size and Trends, 2017 -                           │
│  2025](https://www.grandviewresearch.com/industry-analysis/artificial-intelligence-in-healthcare-market)        │
│                                                                                                                 │
│  - **Autonomous Vehicles:** The rise of self-driving cars is no longer just a pipe dream, with companies like   │
│  Tesla leading the charge in this space. As these vehicles become more commonplace, we'll see reduced traffic   │
│  fatalities and increased efficiency on our roads. [Source: Autonomous Vehicles: Global Market Size and         │
│  Forecast to 2025](https://www.alliedmarketresearch.com/autonomous-vehicles-market)                             │
│                                                                                                                 │
│  💼 **Key Players in the AI Industry** 🌐💡                                                                     │
│  -------------------------------------                          

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 9e3c02c7-cbb1-443e-82ff-30b8b62df7bf                                                                     │
│  Agent: Editor                                                                                                  │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



Tasks are finished in 45.7560970049999 seconds.




╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: feb721e7-d204-4d8a-b557-a869ea16228c                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ```markdown                                                                                      │
│  Artificial Intelligence: Navigating the New Wave of Innovation 🌐💡🔬                                          │
│  ===================================================================                                            │
│                                                                                                                 │
│  Welcome to the cutting edge of technological evolution! Today, we're diving deep into the world of Artificial  │
│  Intelligence (AI), a realm brimming with potential that promises to revolutionize industries and shape our     │
│  future. So buckle up as we traverse the latest trends, key players, impactful applications, and ethical        │
│  considerations in AI technology.                                                                               │
│                                                                                                                 │
│  🌐 **Latest Trends in AI** 🚀💻                                                                                │
│  -----------------------------                                                                                  │
│                                                                                                                 │
│  The AI landscape is evolving at an unprecedented pace, with breakthroughs and advancements that are reshaping  │
│  the way we live and work. From AI-powered healthcare systems that diagnose diseases faster than ever before    │
│  to autonomous vehicles cruising our streets, the impact of AI is becoming increasingly palpable. Here's a      │
│  glimpse at some emerging trends that are steering the course for this transformative technology:               │
│                                                                                                                 │
│  - **AI-Powered Healthcare:** Medical professionals are now leveraging AI to sift through massive amounts of    │
│  data and make accurate diagnoses faster than ever before. The potential applications for this technology in    │
│  the healthcare industry are vast, ranging from personalized treatment plans to predictive analytics that       │
│  could save countless lives. [Source: AI in Healthcare Market Size and Trends, 2017 -                           │
│  2025](https://www.grandviewresearch.com/industry-analysis/artificial-intelligence-in-healthcare-market)        │
│                                                                                                                 │
│  - **Autonomous Vehicles:** The rise of self-driving cars is no longer just a pipe dream, with companies like   │
│  Tesla leading the charge in this space. As these vehicles become more commonplace, we'll see reduced traffic   │
│  fatalities and increased efficiency on our roads. [Source: Autonomous Vehicles: Global Market Size and         │
│  Forecast to 2025](https://www.alliedmarketresearch.com/autonomous-vehicles-market)                             │
│                                                                                                                 │
│  💼 **Key Players in the AI Industry** 🌐💡                    

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ a53bf245-d40f-4f2d-852e-2c68b8ec2edf                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/a53bf245-d40f-4f2 │
│ d-852e-2c68b8ec2edf?access_code=TRACE-d5e9bf62ef                             │
│ 🔑 Access Code: TRACE-d5e9bf62ef                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


- Display the results of your execution as markdown in the notebook.

In [15]:
from IPython.display import Markdown
Markdown(result.raw)

```markdown
Artificial Intelligence: Navigating the New Wave of Innovation 🌐💡🔬
===================================================================

Welcome to the cutting edge of technological evolution! Today, we're diving deep into the world of Artificial Intelligence (AI), a realm brimming with potential that promises to revolutionize industries and shape our future. So buckle up as we traverse the latest trends, key players, impactful applications, and ethical considerations in AI technology.

🌐 **Latest Trends in AI** 🚀💻
-----------------------------

The AI landscape is evolving at an unprecedented pace, with breakthroughs and advancements that are reshaping the way we live and work. From AI-powered healthcare systems that diagnose diseases faster than ever before to autonomous vehicles cruising our streets, the impact of AI is becoming increasingly palpable. Here's a glimpse at some emerging trends that are steering the course for this transformative technology:

- **AI-Powered Healthcare:** Medical professionals are now leveraging AI to sift through massive amounts of data and make accurate diagnoses faster than ever before. The potential applications for this technology in the healthcare industry are vast, ranging from personalized treatment plans to predictive analytics that could save countless lives. [Source: AI in Healthcare Market Size and Trends, 2017 - 2025](https://www.grandviewresearch.com/industry-analysis/artificial-intelligence-in-healthcare-market)

- **Autonomous Vehicles:** The rise of self-driving cars is no longer just a pipe dream, with companies like Tesla leading the charge in this space. As these vehicles become more commonplace, we'll see reduced traffic fatalities and increased efficiency on our roads. [Source: Autonomous Vehicles: Global Market Size and Forecast to 2025](https://www.alliedmarketresearch.com/autonomous-vehicles-market)

💼 **Key Players in the AI Industry** 🌐💡
-------------------------------------

The AI industry is bustling with innovation, and several key players are driving this transformation. Here's a snapshot of some of the companies shaping the future of AI:

- **Google:** Google has been at the forefront of AI research for years, using its resources to develop cutting-edge technology in areas like natural language processing (NLP) and machine learning (ML). Their efforts have resulted in groundbreaking advancements that are improving our daily lives.

- **Tesla:** Tesla's foray into the world of AI has been nothing short of remarkable, with its self-driving car technology pushing the boundaries of what's possible. Elon Musk and his team are relentlessly pursuing a future where autonomous vehicles become the norm.

🏭 **Impact of AI on Various Industries** 💼🌐
----------------------------------------

AI is making its presence felt across various industries, from finance to manufacturing, and retail. Here's how these sectors are benefiting (and sometimes grappling with challenges) due to the adoption of AI:

- **Finance:** AI is being used in the financial sector to streamline processes, improve customer experiences, and make more informed investment decisions. However, there are concerns about data privacy and security as well as potential job displacement. [Source: AI in Finance Market - Growth, Trends, and Forecast (2019 - 2024)](https://www.mordorintelligence.com/industry-reports/ai-in-finance-market)

- **Manufacturing:** AI is helping manufacturers optimize their production processes, reduce waste, and increase efficiency. While these benefits are significant, there's also a need to address concerns about job displacement and the ethical implications of increased automation. [Source: The Future of AI in Manufacturing](https://tractica.com/report/the-future-of-ai-in-manufacturing/)

🔮 **The Future of AI** 🚀💡🌐
---------------------------

As we look to the future, there are several exciting developments on the horizon for AI technology:

- **Advancements in Machine Learning:** We can expect machine learning algorithms to become more sophisticated and capable of handling larger datasets with greater accuracy.

- **Improvements in Natural Language Processing:** NLP will continue to evolve, making it possible for AI systems to understand human language more naturally and accurately.

🌐 **Ethical Considerations** 🔮💡
-------------------------------

As with any powerful technology, AI comes with its share of ethical considerations that must be addressed:

- **Data Privacy:** Ensuring the privacy of individuals' data is paramount as AI systems collect and process increasing amounts of personal information.

- **Bias:** There's a need to address potential biases in AI systems, which can lead to unfair outcomes in areas like criminal justice and hiring practices.

- **Job Displacement:** The rise of automation due to AI may displace certain jobs, raising concerns about the future of work and the need for re-skilling efforts.

💬 **Call to Action** 🚀💻
------------------------

We encourage you to delve deeper into the world of Artificial Intelligence by exploring additional resources on this fascinating topic. Don't hesitate to share your thoughts, experiences, and questions with us in the comments section below!
```